# 19 — Bài báo, bước 3a': lỗi giữ nhầm OTHER của LLM trên Lazada và Hold-out (máy local)

**Câu hỏi**

Vì sao LLM gọi nhầm nhiều review OTHER của Lazada là ACTIONABLE hơn hẳn so với Hold-out Shopee. Thiết kế pre-register ở amendment v3.7;
đây là phân tích khám phá, chỉ dùng dự đoán đã lưu, không gọi API và không đổi mô hình.

**Hai phần**

Phần định lượng dùng các nhóm gợi ý OTHER cố định lấy từ guideline để so thành phần review OTHER giữa hai ứng dụng và tỷ lệ bị gọi
nhầm trong từng nhóm. Phần định tính dựa trên mã hóa thủ công của người nghiên cứu trên phiếu mù (không ghi ứng dụng, không hiện gợi ý).
Notebook không in văn bản review.

In [1]:
import os, sys, json, datetime
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import paper_analysis as PA
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
SHEET_DIR = os.path.join(ROOT, '07_human_eval', 'paper_lazada_errors')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
SETS = {'Hold-out': ('shopee_holdout.csv', 'holdout'), 'Lazada': ('lazada_external_test.csv', 'lazada')}
D, FLAG = {}, {}
for name, (f, key) in SETS.items():
    d = pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'text', 'gold_label'], encoding='utf-8-sig')
    c = pd.read_csv(os.path.join(RES, f'gd10_llm_{key}_tier1.csv')).set_index('review_id').loc[d['review_id']]
    fl = pd.read_csv(os.path.join(RES, f'paper_flat_llm_{key}.csv')).set_index('review_id').loc[d['review_id']]
    d['cascade gọi ACTIONABLE'] = (c['pred_label'] == 'ACTIONABLE').to_numpy()
    d['flat gọi ACTIONABLE'] = (fl['pred_label'] != 'OTHER').to_numpy()
    d['nhãn flat'] = fl['pred_label'].to_numpy()
    D[name], FLAG[name] = d, PA.cue_flags(d['text'])
    other = d['gold_label'] == 'OTHER'
    print(f"{name}: {len(d)} review, {other.sum()} OTHER | giữ nhầm: cascade {int((other & d['cascade gọi ACTIONABLE']).sum())}, "
          f"flat {int((other & d['flat gọi ACTIONABLE']).sum())}")

Hold-out: 1000 review, 817 OTHER | giữ nhầm: cascade 47, flat 70
Lazada: 500 review, 419 OTHER | giữ nhầm: cascade 64, flat 81


## Thành phần review OTHER và tỷ lệ bị gọi nhầm

**Cách đọc**

"% OTHER" là tỷ lệ review OTHER có nhóm gợi ý đó (thành phần); "tỷ lệ giữ nhầm" là phần trong nhóm bị LLM gọi ACTIONABLE (hành vi).
Một review có thể thuộc nhiều nhóm, nên các tỷ lệ thành phần không cộng thành 100%.

In [2]:
TAB = {}
for model in ('cascade', 'flat'):
    parts = []
    for name, d in D.items():
        t = PA.kept_other_table(FLAG[name], (d['gold_label'] == 'OTHER').to_numpy(), d[f'{model} gọi ACTIONABLE'].to_numpy())
        parts.append(t.set_index('nhóm').add_suffix(f' — {name}'))
    TAB[model] = pd.concat(parts, axis=1).reset_index()
    cols = ['nhóm'] + [f'{c} — {n}' for c in ('% OTHER', 'tỷ lệ giữ nhầm (%)', 'n giữ nhầm') for n in D]
    print(f'LLM {model} v1:'); print(TAB[model][cols].round(1).to_string(index=False)); print()

LLM cascade v1:
                           nhóm  % OTHER — Hold-out  % OTHER — Lazada  tỷ lệ giữ nhầm (%) — Hold-out  tỷ lệ giữ nhầm (%) — Lazada  n giữ nhầm — Hold-out  n giữ nhầm — Lazada
                   tất cả OTHER               100.0             100.0                            5.8                         15.3                     47                   64
                      giao hàng                13.0              17.2                            1.9                          4.2                      2                    3
            người bán, sản phẩm                 8.3              12.2                            5.9                          3.9                      4                    2
              giá, phí, voucher                11.1              21.7                           20.9                         22.0                     19                   20
chính sách, tài khoản, đơn hàng                13.5              19.8                           11.8              

## Phiếu mã hóa thủ công

**Phạm vi**

Mọi review gold OTHER bị cascade LLM v1 hoặc flat LLM v1 gọi ACTIONABLE, trên cả Lazada và Hold-out.

**Phiếu mù**

`coding_sheet.csv` chỉ có mã dòng và văn bản, xếp ngẫu nhiên (seed 42), không ghi ứng dụng và không hiện gợi ý từ khóa; tập, dự đoán và
nhóm gợi ý nằm trong `coding_key.csv`. Hướng dẫn và codebook ở `README.md` cùng thư mục. Phiếu đã có mã hóa thì không bị ghi đè.

In [3]:
rows = []
for name, d in D.items():
    m = (d['gold_label'] == 'OTHER') & (d['cascade gọi ACTIONABLE'] | d['flat gọi ACTIONABLE'])
    k = d[m].copy()
    f = FLAG[name][m.to_numpy()]
    k['tập'] = name
    k['bị gọi ACTIONABLE bởi'] = np.select([k['cascade gọi ACTIONABLE'] & k['flat gọi ACTIONABLE'], k['cascade gọi ACTIONABLE']],
                                           ['cả hai', 'cascade'], 'flat')
    k['nhóm gợi ý (từ khóa)'] = [', '.join(c for c in PA.OTHER_CUES if r[c]) or 'không nhóm nào' for _, r in f.iterrows()]
    k['nhắc app hoặc lỗi'] = f['nhắc app hoặc lỗi'].to_numpy()
    rows.append(k)
KEY = pd.concat(rows, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
KEY.insert(0, 'mã dòng', [f'R{i + 1:03d}' for i in range(len(KEY))])
print('Số review cần mã hóa:', KEY.groupby(['tập', 'bị gọi ACTIONABLE bởi']).size().to_dict(), '| tổng', len(KEY))
os.makedirs(SHEET_DIR, exist_ok=True)
SHEET = os.path.join(SHEET_DIR, 'coding_sheet.csv')
KEYF = os.path.join(SHEET_DIR, 'coding_key.csv')
sheet = KEY[['mã dòng', 'text']].assign(**{'nhóm chính': '', 'ranh giới (có/không)': '', 'ghi chú': ''})
if os.path.exists(SHEET):
    old = pd.read_csv(SHEET, encoding='utf-8-sig', dtype=str).fillna('')
    assert old['mã dòng'].tolist() == sheet['mã dòng'].tolist() and old['text'].tolist() == sheet['text'].astype(str).tolist(), \
        'Phiếu cũ khác danh sách hiện tại — dừng, không ghi đè.'
    print('Phiếu đã tồn tại, giữ nguyên; số dòng đã mã hóa:', int((old['nhóm chính'] != '').sum()))
else:
    sheet.to_csv(SHEET, index=False, encoding='utf-8-sig')
    print('Đã tạo phiếu:', os.path.relpath(SHEET, ROOT))
KEY.drop(columns=['text']).to_csv(KEYF, index=False, encoding='utf-8-sig')
print('Đã ghi file khóa:', os.path.relpath(KEYF, ROOT))

Số review cần mã hóa: {('Hold-out', 'cascade'): 5, ('Hold-out', 'cả hai'): 42, ('Hold-out', 'flat'): 28, ('Lazada', 'cascade'): 5, ('Lazada', 'cả hai'): 59, ('Lazada', 'flat'): 22} | tổng 161
Phiếu đã tồn tại, giữ nguyên; số dòng đã mã hóa: 161
Đã ghi file khóa: 07_human_eval\paper_lazada_errors\coding_key.csv


## Kết quả mã hóa

**Điều kiện**

Chỉ chạy khi mọi dòng của phiếu đã có "nhóm chính". Bảng gồm phân bố nhóm chính của lỗi giữ nhầm theo từng tập và từng mô hình, tỷ lệ
dòng được đánh dấu ranh giới, độ khớp giữa nhóm chính của người với nhóm gợi ý từ khóa, và đóng góp của từng nhóm vào tỷ lệ giữ nhầm
(số lỗi của nhóm chia cho số review OTHER của tập) để so phần chênh giữa hai ứng dụng.

In [4]:
CODES = ['giao hàng', 'người bán, sản phẩm', 'giá, phí, voucher', 'chính sách, tài khoản, đơn hàng', 'chăm sóc khách hàng',
         'than phiền app chung chung', 'khen chung chung', 'khác']
old = pd.read_csv(SHEET, encoding='utf-8-sig', dtype=str).fillna('')
done = old['nhóm chính'].str.strip() != ''
if not done.all():
    print(f'Chưa đủ mã hóa: {int(done.sum())}/{len(old)} dòng — bỏ qua phần kết quả.')
else:
    bad = sorted(set(old['nhóm chính'].str.strip()) - set(CODES))
    assert not bad, f'Nhóm ngoài codebook: {bad}'
    C = KEY.drop(columns=['text']).merge(old[['mã dòng', 'nhóm chính', 'ranh giới (có/không)']], on='mã dòng')
    C['nhóm chính'] = C['nhóm chính'].str.strip()
    out = []
    for model in ('cascade', 'flat'):
        sub = C[C[f'{model} gọi ACTIONABLE']]
        dist = (sub.groupby('tập')['nhóm chính'].value_counts(normalize=True).mul(100).unstack(0).reindex(CODES).fillna(0))
        dist.columns = [f'{model} — {c} (%)' for c in dist.columns]
        out.append(dist)
    DIST = pd.concat(out, axis=1).round(1)
    print('Phân bố nhóm chính của lỗi giữ nhầm (%):'); print(DIST.to_string())
    border = C.groupby('tập')['ranh giới (có/không)'].apply(lambda s: (s.str.strip().str.lower() == 'có').mean() * 100).round(1)
    print('\nTỷ lệ dòng được đánh dấu ranh giới (%):', border.to_dict())
    has_cue = C['nhóm chính'].isin(list(PA.OTHER_CUES))
    hit = [c in s for c, s in zip(C.loc[has_cue, 'nhóm chính'], C.loc[has_cue, 'nhóm gợi ý (từ khóa)'])]
    print(f'Nhóm chính của người nằm trong nhóm gợi ý từ khóa: {np.mean(hit) * 100:.1f}% ({int(np.sum(hit))}/{len(hit)} dòng có nhóm tương ứng)')
    DIST.reset_index().to_csv(os.path.join(RES, 'paper_lazada_errors_human_coding.csv'), index=False, encoding='utf-8-sig')
    n_other = pd.Series({name: int((d['gold_label'] == 'OTHER').sum()) for name, d in D.items()})
    parts = []
    for model in ('cascade', 'flat'):
        cnt = C[C[f'{model} gọi ACTIONABLE']].groupby(['tập', 'nhóm chính']).size().unstack(0).reindex(CODES).fillna(0)[list(D)]
        rate = cnt.div(n_other[list(D)], axis=1) * 100
        rate.loc['tổng'] = rate.sum()
        rate[f'chênh'] = rate['Lazada'] - rate['Hold-out']
        rate.columns = [f'{model} — {c}' for c in ('Hold-out (% OTHER)', 'Lazada (% OTHER)', 'chênh Lazada − Hold-out (điểm %)')]
        parts.append(rate)
    CONTRIB = pd.concat(parts, axis=1).round(2)
    print('\nĐóng góp của từng nhóm vào tỷ lệ giữ nhầm (số lỗi của nhóm / số review OTHER của tập, %):'); print(CONTRIB.to_string())
    CONTRIB.reset_index().to_csv(os.path.join(RES, 'paper_lazada_errors_contribution.csv'), index=False, encoding='utf-8-sig')
for model, t in TAB.items():
    t.to_csv(os.path.join(RES, f'paper_lazada_errors_cues_{model}.csv'), index=False, encoding='utf-8-sig')

Phân bố nhóm chính của lỗi giữ nhầm (%):
                                 cascade — Hold-out (%)  cascade — Lazada (%)  flat — Hold-out (%)  flat — Lazada (%)
nhóm chính                                                                                                           
giao hàng                                           0.0                   1.6                  0.0                4.9
người bán, sản phẩm                                10.6                   0.0                 11.4                2.5
giá, phí, voucher                                  23.4                  14.1                 17.1                9.9
chính sách, tài khoản, đơn hàng                    46.8                  76.6                 45.7               76.5
chăm sóc khách hàng                                 4.3                   0.0                  0.0                0.0
than phiền app chung chung                         14.9                   4.7                 24.3                3.7
khen chung chun

## Nhận xét

**Mã hóa thủ công**

161 review bị LLM gọi nhầm được người nghiên cứu mã hóa trên phiếu mù, không biết review thuộc ứng dụng nào và không thấy gợi ý từ khóa.
Mọi dòng đều có nhóm chính hợp lệ.

**Kết quả chính**

Lỗi giữ nhầm trên Lazada dồn vào nhóm chính sách, tài khoản, đơn hàng: 76,6% lỗi của cascade LLM và 76,5% của flat LLM, so với 46,8% và
45,7% trên Hold-out. Quy về tỷ lệ trên toàn bộ review OTHER, nhóm này chiếm 9,00 trong 9,52 điểm chênh tỷ lệ giữ nhầm giữa hai ứng dụng
với cascade, và 10,88 trên 10,76 điểm với bản phẳng (các nhóm còn lại bù trừ nhau).

**Diễn giải**

Review than phiền về kết quả của chính sách hay quy trình (tài khoản bị khóa, đơn bị hủy, hoàn tiền, trả hàng) được guideline xếp vào OTHER
theo quy tắc R6, nhưng LLM thường coi chúng là vấn đề của app. Review Lazada có nhiều than phiền loại này hơn và LLM gọi nhầm chúng nhiều hơn,
nên điểm yếu của LLM khi sang ứng dụng mới nằm ở việc áp quy tắc R6.

**Ca ranh giới**

Chỉ 10,7% lỗi trên Hold-out và 12,8% trên Lazada được người mã hóa đánh dấu là ca ranh giới, nên phần lớn là lỗi thật của LLM so với
guideline chứ không phải do nhãn gold mơ hồ.

**Nhóm từ khóa**

Nhóm chính của người nằm trong nhóm gợi ý từ khóa ở 56,5% dòng có nhóm tương ứng, nên phân tích từ khóa chỉ là chỉ báo thô; kết luận dựa
trên mã hóa của người.

**Giới hạn và hướng tiếp**

Chỉ có một người mã hóa, cũng là người gán nhãn gold, nên không đo được độ đồng thuận. Thêm ví dụ few-shot minh họa quy tắc R6 là một hướng
cải thiện, nhưng cần được đánh giá trên dữ liệu độc lập chứ không chỉnh trên Lazada.